# ISO 20414:2020 — Test 16: Affiliation to Familiar Exits

A room of size 10 m x 15 m with two 1 m exits on the 15 m walls of the room (ISO 20414:2020 Figure 12). In Scenario 1, the agent, equidistant to both exits, in the room and is free to choose between exits 1 and 2 (both exit weights are 50%). After statistics harvested for exit choice through several runs, Scenario 2 is run in which the same occupant is affiliated with Exit 2. Test is run several times again. Expected result is that the usage of Exit 2 is increased in scenario 2.

In [1]:
from datetime import datetime
print(f"Executed on {datetime.now().astimezone().strftime('%d %B %Y, %H:%M %Z')}")

Executed on 14 July 2026, 12:52 CEST


In [2]:
from pathlib import Path
import logging
logging.basicConfig(level=logging.WARNING, force=True)
logging.getLogger().setLevel(logging.WARNING)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pedpy
from shapely.geometry import Point, Polygon

from jupedsim_scenarios import load_scenario, run_scenario, run_sweep

In [3]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f7f7f5",
    "axes.edgecolor": "#3a3a3a",
    "axes.labelcolor": "#1d1d1d",
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.figsize": (8, 5),
})

## Load and Run

In [4]:
%%capture
SCENARIO_ZIP = Path("scenario_files") / "Iso-16-familiar-exits.zip"
scenario = load_scenario(str(SCENARIO_ZIP))
result = run_scenario(scenario, seed=42)

In [5]:
def count_exit_choices(scenario, seeds, workers=4):
    sweep = run_sweep(scenario, seeds=seeds, workers=workers)

    exit_1_count = 0
    exit_2_count = 0
    unknown_count = 0
    rows = []

    for trial in sweep.trials:
        seed = trial.seed
        result = trial.result
        df_1 = result.trajectory_dataframe()
        final_rows = [df_1.sort_values("frame").iloc[-1]]

        for last in final_rows:
            if last["y"] >= 11.0:
                if last["x"] <= 1.5:
                    chosen_exit = "exit_1"
                    exit_1_count += 1
                elif last["x"] >= 8.5:
                    chosen_exit = "exit_2"
                    exit_2_count += 1
                else:
                    chosen_exit = "unknown"
                    unknown_count += 1
            else:
                chosen_exit = "no exit"
                unknown_count += 1

            rows.append({
                "seed": seed,
                "id": int(last["id"]),
                "x_final": last["x"],
                "y_final": last["y"],
                "chosen_exit": chosen_exit,
            })

    sweep.cleanup()

    exit_choices = pd.DataFrame(rows)
    total = len(exit_choices)

    return {
        "exit_1_count": exit_1_count,
        "exit_2_count": exit_2_count,
        "unknown_count": unknown_count,
        "exit_1_fraction": exit_1_count / total,
        "exit_2_fraction": exit_2_count / total,
        "unknown_fraction": unknown_count / total,
        "rows": exit_choices,
    }

## Scenario 1 Statistics

In [6]:
run_1a = count_exit_choices(scenario, seeds=range(1, 4001), workers=4)
run_1b = count_exit_choices(scenario, seeds=range(4001, 8001), workers=4)
comparison = pd.DataFrame([
    {
        "exit": "exit_1",
        "run_1a_count": run_1a["exit_1_count"],
        "run_1b_count": run_1b["exit_1_count"],
        "run_1a_fraction": run_1a["exit_1_fraction"],
        "run_1b_fraction": run_1b["exit_1_fraction"],
    },
    {
        "exit": "exit_2",
        "run_1a_count": run_1a["exit_2_count"],
        "run_1b_count": run_1b["exit_2_count"],
        "run_1a_fraction": run_1a["exit_2_fraction"],
        "run_1b_fraction": run_1b["exit_2_fraction"],
    },
])

comparison


,exit,run_1a_count,run_1b_count,run_1a_fraction,run_1b_fraction
0,exit_1,1923,1911,0.48075,0.47775
1,exit_2,2077,2089,0.51925,0.52225


## Acceptance - Scenario 1

In [7]:
assert abs(run_1a["exit_1_fraction"] - run_1b["exit_1_fraction"]) * 100 <= 1.0
assert abs(run_1a["exit_2_fraction"] - run_1b["exit_2_fraction"]) * 100 <= 1.0

## Scenario 2 Statistics

In [ ]:
DIST_ID = "jps-distributions_0"
JOURNEY_R1 = "journey-1781167866144-fp65t3"
JOURNEY_R2 = "journey-1781167890022-3hfgs4"
WORKERS=4
SEEDS_PER_COND=2

def set_journey_weight(scenario, dist_id, journey_id, weight):
    for entry in scenario.raw["distributions"][dist_id]["journey_weights"]:
        if entry["journey_id"] == journey_id:
            entry["weight"] = weight
            return

def classify_exit(row):
    """Infer exit from final x position (from your config.json coordinates)."""
    if row["x"] <= 1.5:
        return "exit_1"
    elif row["x"] >= 8.5:
        return "exit_2"
    return "unknown"

sweep = run_sweep(
    scenario,
    axes={"route2_weight": [50, 60, 70, 80, 90, 100]},
    apply={
        "route2_weight": lambda s, w: (
            set_journey_weight(s, DIST_ID, JOURNEY_R1, 100 - w),
            set_journey_weight(s, DIST_ID, JOURNEY_R2, w),
        )
    },
    seeds=range(100, 100 + SEEDS_PER_COND),
    workers=WORKERS,
)

#  Build per-agent exit choice table 
rows = []
for trial in sweep.trials:
    df_traj = trial.result.trajectory_dataframe()
    # Last recorded frame per agent = where they exited
    last = df_traj.sort_values("frame").groupby("id").last().reset_index()
    last["exit"] = last.apply(classify_exit, axis=1)
    last["route2_weight"] = trial.axis_values["route2_weight"]
    last["seed"] = trial.seed
    rows.append(last)

df_exits = pd.concat(rows, ignore_index=True)

# --- Aggregate: mean fraction going to each exit per weight condition ---
agg = (
    df_exits.groupby(["route2_weight", "exit"])
    .size()
    .reset_index(name="count")
    .pivot(index="route2_weight", columns="exit", values="count")
    .fillna(0)
    .reset_index()
)
agg.columns.name = None
agg["total"] = agg[["exit_1", "exit_2"]].sum(axis=1)
agg["exit_1_frac"] = agg["exit_1"] / agg["total"]
agg["exit_2_frac"] = agg["exit_2"] / agg["total"]

agg 


SyntaxError: invalid syntax (3574489301.py, line 10)

## Plot Trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(agg["route2_weight"], agg["exit_1_frac"], "o-", label="Exit 1 (non-familiar)", color="#1f6feb")
ax.plot(agg["route2_weight"], agg["exit_2_frac"], "o-", label="Exit 2 (familiar/affiliated)", color="#e05c2a")
ax.axvline(50, color="gray", linestyle="--", alpha=0.5, label="Scenario 1 (50/50)")
ax.axvline(80, color="green", linestyle="--", alpha=0.5, label="Scenario 2 (affiliated)")
ax.set_xlabel("Route 2 weight")
ax.set_ylabel("Fraction of agents")
ax.set_title("ISO 20414 Test 16 — Exit Choice vs Affiliation Weight")
ax.legend()
plt.tight_layout()
plt.show()

## Acceptance - Scenario 2

In [ ]:
sweep.cleanup()
scen1 = agg.loc[agg["route2_weight"] == 50].iloc[0]
scen2 = agg.loc[agg["route2_weight"] == 80].iloc[0]

assert abs(scen1["exit_1_frac"] - scen1["exit_2_frac"]) <= 0.10, \
    f"FAIL Scenario 1: exits not balanced (exit_1={scen1['exit_1_frac']:.0%}, exit_2={scen1['exit_2_frac']:.0%})"

# Scenario 2: with affiliation to exit 2, exit 2 usage must be strictly greater than exit 1
assert scen2["exit_2_frac"] > scen2["exit_1_frac"], \
    f"FAIL Scenario 2: exit 2 not preferred (exit_1={scen2['exit_1_frac']:.0%}, exit_2={scen2['exit_2_frac']:.0%})"

print(f"   Scenario 1 — exit_1: {scen1['exit_1_frac']:.0%}  exit_2: {scen1['exit_2_frac']:.0%}  (balanced)")
print(f"   Scenario 2 — exit_1: {scen2['exit_1_frac']:.0%}  exit_2: {scen2['exit_2_frac']:.0%}  (affiliation confirmed)")